# Session 2 - Lab: build an autoencoder, then find out what it is worth

**DAT09-06 Representation Learning · Monday 9 November 2026 · 1 h 20**

You are building the first piece of the assistant's memory: a map that turns a
product image into a handful of numbers, learned without a single label.

### How this notebook is marked

| Marker | What it means |
|---|---|
| `# Given - read it, run it, move on.` | nothing to write. Run the cell. |
| ✏️ **YOUR TURN** | a boxed banner tells you **how many lines** and **what they do**. That is the only place you write code. |
| ❓ **Checkpoint** | questions about *your* output. Answer **in writing**, in the cell below. |

There are **12 cells to write in the whole notebook**, and none of them asks
for more than six lines of code. Everything else is given.

### Read this before you start

An AI will write all twelve in about two minutes. Let it.

It cannot run your experiment, read your numbers, or decide what they mean -
and that is what this lab is made of. **The questions are the lab. The code is
the easy part.**

### Six checkpoints

| | What you produce | ~ |
|---|---|---|
| **1** | An autoencoder that trains, and proof it is not cheating | 15 min |
| **2** | Where the bottleneck spends its budget, class by class | 15 min |
| **3** | The four-number verdict, and the mistakes the probe makes | 20 min |
| **4** | A representation wrecked by something irrelevant | 20 min |
| **5** | The holes in the latent space | 15 min |
| **6** | **The challenge** - best representation under a hard budget | open |

**Most of you will not reach checkpoint 6. That is deliberate.** Finish the
questions in 1-5 properly rather than rushing to the end with nothing to say.

Run the cells **in order**.

---

## 0 · Setup

Two cells, both given. If the first one fails you are not in the repository
folder, or the environment is not activated.

In [ ]:
# Given - read it, run it, move on.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # make `common` importable

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from common.evaluation import compare_reports, evaluate_representation
from common.paths import PROCESSED, require
from common.plotting import PALETTE, use_course_style
from common.seeding import SEED, cpu_threads, set_seed

use_course_style()
set_seed(SEED)
cpu_threads()

print("torch", torch.__version__, "| device: cpu")

### 0.2 · The catalogue

Fashion-MNIST: 60,000 product photos, 28x28, grayscale.

The last 6,000 are held out. **Every number we report is computed on those**,
never on the images the model trained on. The labels are loaded too - and then
set aside. They are the ruler, not the training signal.

In [ ]:
# Given - read it, run it, move on.
blob = np.load(require(PROCESSED / "fmnist_train.npz",
                       hint="python data/prepare_data.py --fmnist"))
images, labels = blob["images"], blob["labels"]
class_names = list(blob["classes"])

x = images.reshape(len(images), -1).astype(np.float32) / 255.0   # (60000, 784)

x_train_np, x_val_np = x[:54000], x[54000:]
y_train, y_val = labels[:54000], labels[54000:]

x_train = torch.from_numpy(x_train_np.copy())
x_val = torch.from_numpy(x_val_np.copy())

print(f"train {tuple(x_train.shape)}   val {tuple(x_val.shape)}")
print("classes:", ", ".join(class_names))

---

# Checkpoint 1 - Make it train, then prove it is not cheating

**~15 minutes.** Three cells to write, two lines each.

### 1.1 · The shape we are building

$$784 \;\xrightarrow{\;f_\phi\;}\; 256 \;\xrightarrow{\;\mathrm{ReLU}\;}\; d
\;\xrightarrow{\;g_\theta\;}\; 256 \;\xrightarrow{\;\mathrm{ReLU}\;}\; 784$$

Here is the **encoder**, given. It ends at $d$ with no activation: the code is
free to take any value.

In [ ]:
# Given - read it, run it, move on.
def encoder_for(d, hidden=256):
    return nn.Sequential(
        nn.Linear(784, hidden),
        nn.ReLU(),
        nn.Linear(hidden, d),
    )


print(encoder_for(32))

### 1.2 ✏️ · The decoder is its mirror image

One difference: it ends with a **Sigmoid**, because our pixels live in
$[0, 1]$ and we want the output to live there too.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 4 lines, inside nn.Sequential
#     mirror encoder_for: d -> hidden -> 784
#     ReLU in the middle, Sigmoid at the very end
# ──────────────────────────────────────────────────────────────────────
def decoder_for(d, hidden=256):
    ...          # <- replace this line


print(decoder_for(32))

### 1.3 · Wrapping the two together

Given. Note that `forward` returns **both** the reconstruction and the code -
you need $z$ in checkpoint 3, and running the encoder twice would be a waste.

In [ ]:
# Given - read it, run it, move on.
class Autoencoder(nn.Module):
    """784 -> 256 -> d -> 256 -> 784."""

    def __init__(self, d: int, hidden: int = 256):
        super().__init__()
        self.d = d
        self.encoder = encoder_for(d, hidden)
        self.decoder = decoder_for(d, hidden)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

### 1.4 ✏️ · One training step

This is the heart of it, and it is four lines. Everything else in the training
loop is bookkeeping.

> **Watch out.** Forgetting `zero_grad()` makes gradients accumulate across
> batches. The loss still goes down, just worse, and nothing tells you. It is
> the most common silent bug in this course.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 5 lines
#     1. forward: run the model on x_batch, keep the reconstruction
#     2. loss:    loss_fn(reconstruction, x_batch) - the target IS the input
#     3. opt.zero_grad(), then loss.backward()
#     4. opt.step()
# ──────────────────────────────────────────────────────────────────────
def one_step(model, x_batch, opt, loss_fn):
    """Forward, loss, backward, nudge. Returns the loss as a float."""
    ...          # <- replace this line
    return loss.item()

### 1.5 · The loop around it

Given, and you will not touch it again: every model in this notebook is trained
by this one function.

In [ ]:
# Given - read it, run it, move on.
def train(model, data, *, epochs=8, batch=256, lr=1e-3, verbose=True):
    """Train an autoencoder on `data`, return the per-epoch loss history."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    n, history = len(data), []

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        running = 0.0

        for i in range(0, n, batch):
            x_batch = data[perm[i:i + batch]]
            running += one_step(model, x_batch, opt, loss_fn) * len(x_batch)

        history.append(running / n)
        if verbose:
            print(f"  epoch {epoch + 1:>2}/{epochs}   mse = {history[-1]:.5f}")

    return history

### 1.6 ▶ · Train one, and watch the loss fall

About 90 seconds. Nothing to write.

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED)
model32 = Autoencoder(d=32)
history = train(model32, x_train, epochs=8)

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(range(1, len(history) + 1), history, marker="o", color=PALETTE["accent"])
ax.set_xlabel("epoch"); ax.set_ylabel("training MSE")
ax.set_title("Checkpoint 1 - the loss goes down")
plt.show()

### 1.7 ✏️ · Is that loss any good?

A number on its own means nothing. Before you can say your autoencoder works,
you need to know what a model that learned **nothing** would score.

The laziest possible model ignores its input and always outputs the **average
training image**. That is the ceiling of stupidity - your model has to be far
below it.

In [ ]:
# Given - read it, run it, move on.
@torch.no_grad()
def val_mse(model):
    """Reconstruction MSE on the held-out products."""
    model.eval()
    x_hat, _ = model(x_val)
    return float(((x_hat.numpy() - x_val_np) ** 2).mean())


mean_image = x_train_np.mean(axis=0)

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line of numpy
#     the MSE you get by always predicting `mean_image`,
#     measured on x_val_np. No model involved.
# ──────────────────────────────────────────────────────────────────────
lazy_mse = ...          # <- replace this line

ours = val_mse(model32)
print(f"always-the-mean-image : {lazy_mse:.5f}")
print(f"our autoencoder d=32  : {ours:.5f}")
print(f"gap to zero closed    : {100 * (1 - ours / lazy_mse):.1f}%")

### ❓ Checkpoint 1

1. Write down your three numbers. What percentage of the way from "always the
   mean image" to a perfect reconstruction did your model get?
2. Suppose I set `d = 784` and deleted both hidden layers, so the network is
   `Linear(784, 784)` then `Linear(784, 784)`. What validation MSE would you
   expect? **Why would that be bad news rather than good news?**
3. Your training loss is computed on images the model trained on. `val_mse` is
   not. Which of the two would you quote to someone deciding whether to use
   your encoder, and why?

*Your answers:*

1.
2.
3.

---

# Checkpoint 2 - Where the bottleneck spends its budget

**~15 minutes.** Two cells to write, two lines each.

### 2.1 ▶ · Four widths, about ten seconds each

In [ ]:
# Given - read it, run it, move on.
DIMS = [2, 8, 32, 128]
models = {}

for d in DIMS:
    set_seed(SEED)                  # same start every time: a fair comparison
    models[d] = Autoencoder(d=d)
    train(models[d], x_train, epochs=8, verbose=False)
    print(f"d = {d:<4d} val mse = {val_mse(models[d]):.5f}")

### 2.2 ✏️ · Get the reconstructions out

`@torch.no_grad()` is not optional: without it PyTorch builds a gradient graph
for every pixel it draws, and `.numpy()` then refuses to convert a tensor that
still carries one. The decorator is already written for you.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     run the model, keep the reconstruction (not the code),
#     and return it as numpy
# ──────────────────────────────────────────────────────────────────────
@torch.no_grad()
def reconstruct(model, x):
    """Reconstructions as a numpy array of shape (n, 784)."""
    model.eval()
    ...          # <- replace this line


print(reconstruct(models[32], x_val[:4]).shape)   # expect (4, 784)

### 2.3 ▶ · The same product at four widths

In [ ]:
# Given - read it, run it, move on.
n_show = 8
rows = [("original", x_val_np[:n_show])] + [
    (f"d = {d}", reconstruct(models[d], x_val[:n_show])) for d in DIMS
]

fig, axes = plt.subplots(len(rows), n_show, figsize=(n_show * 1.1, len(rows) * 1.2),
                         squeeze=False)
for r, (label, grid) in enumerate(rows):
    for c in range(n_show):
        axes[r][c].imshow(1 - grid[c].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
    axes[r][0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=11)
fig.tight_layout()
plt.show()

### 2.4 ✏️ · Which products does it fail?

Looking at eight pictures is an anecdote. Measure it: the mean reconstruction
error **per class**, at the tightest and at a comfortable width.

This is where the labels earn their keep - not to train anything, only to slice
the error.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     per-image squared error, averaged over the 784 pixels,
#     then averaged within each class. Return an array of length 10.
# ──────────────────────────────────────────────────────────────────────
def per_class_mse(model, x_t, x_np, y):
    """Mean reconstruction MSE for each of the 10 classes."""
    ...          # <- replace this line


mse_2 = per_class_mse(models[2], x_val, x_val_np, y_val)
mse_32 = per_class_mse(models[32], x_val, x_val_np, y_val)
print(mse_2.round(4))

### 2.5 ▶ · The chart

In [ ]:
# Given - read it, run it, move on.
order = np.argsort(-mse_2)
pos = np.arange(10)

fig, ax = plt.subplots(figsize=(8.4, 3.8))
ax.bar(pos - 0.2, mse_2[order], width=0.4, label="d = 2", color=PALETTE["accent"])
ax.bar(pos + 0.2, mse_32[order], width=0.4, label="d = 32", color=PALETTE["teal"])
ax.set_xticks(pos)
ax.set_xticklabels([class_names[k] for k in order], rotation=35, ha="right")
ax.set_ylabel("mean reconstruction MSE")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

for k in order:
    print(f"  {class_names[k]:<12s} d=2 {mse_2[k]:.4f}   d=32 {mse_32[k]:.4f}")

### ❓ Checkpoint 2

1. Which **three** classes are reconstructed worst at $d = 2$? Which three are
   reconstructed best? Look at the images in those classes - what do the hard
   ones have in common?
2. Going from $d = 2$ to $d = 32$, which class improves the **most**, in
   relative terms? Does that match what you saw in the picture grid?
3. The network was never told what a class is. Yet the error is clearly not
   spread evenly across them. **Where does that structure come from?**
4. At $d = 2$, find one product that comes back as a *different kind of
   product*. Which two classes get confused - and would a shopper make the same
   mistake?

*Your answers:*

1.
2.
3.
4.

---

# Checkpoint 3 - The four-number verdict

**~20 minutes.** Three cells to write.

A reconstruction you like the look of is not a result. This is.

> **train → freeze → probe → compare**

The encoder is already trained, and it never saw a label. Now we freeze it,
read off the codes, and only *then* bring the labels in - to measure.

| Baseline | Why it is the right one |
|---|---|
| **784 raw pixels** | no learning at all. If you cannot beat this, you learned nothing. |
| **PCA at the same $d$** | the same compression, done linearly. The honest bar. |

### First, all four on eight products

Before the real catalogue, the toy space from the slides: eight products with
**two** numbers each, so you can draw it. Four T-shirts at (1,1), (1,3), (3,1),
(3,3) and four ankle boots at (6,1), (6,3), (7,2) - plus **one boot at (2,2)**,
right in the middle of the T-shirts. Run the next cell and watch the numbers
disagree about that one product.

In [ ]:
# Given - read it, run it, move on.
# The eight-product toy space from the slides.
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, silhouette_samples,
                             silhouette_score)

z_toy = np.array([[1., 1.], [1., 3.], [3., 1.], [3., 3.],     # T-shirts
                  [2., 2.], [6., 1.], [6., 3.], [7., 2.]])    # boots, (2,2) is the stray
y_toy = np.array([0, 0, 0, 0, 1, 1, 1, 1])                    # 0 = shirt, 1 = boot

# probe: one logistic regression on the codes
probe = LogisticRegression().fit(z_toy, y_toy).score(z_toy, y_toy)

# 1-NN, leave-one-out: each product's nearest OTHER product
dist = np.linalg.norm(z_toy[:, None] - z_toy[None], axis=-1) + np.eye(8) * 1e9
knn = (y_toy[dist.argmin(1)] == y_toy).mean()

# clustering, with the labels thrown away
C_toy = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(z_toy)

print(f"probe      {probe:.2f}")
print(f"1-NN       {knn:.2f}")
print(f"silhouette {silhouette_score(z_toy, y_toy):.2f}")
print(f"ARI        {adjusted_rand_score(y_toy, C_toy):.2f}")
print()
print("silhouette, product by product:", np.round(silhouette_samples(z_toy, y_toy), 2))
print("the stray boot is the negative one: it sits inside the wrong class.")

plt.figure(figsize=(5, 2.6))
for cls, marker, colour, name in [(0, "o", "#E2643B", "T-shirt"),
                                  (1, "^", "#2F7A80", "ankle boot")]:
    pts = z_toy[y_toy == cls]
    plt.scatter(pts[:, 0], pts[:, 1], marker=marker, c=colour, s=90, label=name)
plt.scatter(*z_toy[4], s=320, facecolors="none", edgecolors="#C0435E", linewidths=2)
plt.legend(loc="upper right", fontsize=8)
plt.title("the ringed boot is the one in the wrong place", fontsize=10)
plt.tight_layout()
plt.show()


The probe says 0.88 and 1-NN says 0.38, on the *same* eight codes. Neither is
wrong. The probe only needs one straight cut to work, while 1-NN is wrecked by a
single product in the wrong neighbourhood: that boot is wrong itself *and* drags
all four T-shirts down with it, because it is the nearest thing to every one of
them.

Keep that in mind when you read your own table in a minute.

### The four numbers, in full

Same numbering as the slides, so you can look them up. $z_i$ is product $i$'s
code, $y_i$ its true class, $d$ the width of the bottleneck.

| | metric | the question it asks | range |
|---|---|---|---|
| 1 | `probe_accuracy` | can one straight line read the classes? | 0 to 1 |
| 2 | `knn_accuracy` | are your neighbours your own kind? | 0 to 1 |
| 3 | `silhouette` | is there a gap between the classes? | $-1$ to $+1$ |
| 4 | `ari` | do two products land in the same group? | 0 to 1 |

The report also carries an `nmi` field. It scores the same clustering as `ari`,
in bits rather than in pairs. **We are not covering it today** - ignore the
column, or read up on normalised mutual information if you are curious.

**1 · `probe_accuracy`** - fit the single matrix $W$ (size $10 \times d$) that best
names 4,200 frozen codes, then measure the share of the other 1,800 it gets
right. Logistic regression and nothing more: it cannot learn a feature, so if it
succeeds, the codes already held the answer.

**2 · `knn_accuracy`** - no model is fitted at all. For each held-out code, take the
$k = 5$ nearest codes by straight-line distance and give it the class that wins
the vote. The share correct. *This is the one a retrieval system feels*, because
the assistant returns neighbours, not a classifier's output.

**3 · `silhouette`** - pure geometry, no labels used for fitting. For product $i$,
let $a_i$ be its mean distance to products of its own class and $b_i$ its mean
distance to those of the nearest other class. Then
$s_i = (b_i - a_i) / \max(a_i, b_i)$, and the silhouette is the average of $s_i$.
In $[-1, 1]$: 1 = deep inside its own class, 0 = sitting on the border.

**4 · `ari`** - run $k$-means (the standard clustering algorithm: give
it $k$, it hunts for $k$ groups of nearby points) with $k = 10$ and **no labels
at all**. It returns a grouping $C$; $Y$ is the truth. ARI scores $C$ against
$Y$: 0 for a random grouping, 1 for a perfect one.

It counts *pairs*. A pair of products is concordant when $C$ and $Y$ agree about
it - together in both, or apart in both. The Rand index RI is the share of
concordant pairs; because most pairs are apart in both, random groupings already
score high, so $\mathrm{ARI} = (\mathrm{RI} - \mathbb{E}[\mathrm{RI}]) /
(1 - \mathbb{E}[\mathrm{RI}])$, where $\mathbb{E}[\mathrm{RI}]$ is what chance gives.

### 3.1 ✏️ · Read the codes off the frozen encoder

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line
#     run ONLY the encoder - not the decoder - and return numpy
# ──────────────────────────────────────────────────────────────────────
@torch.no_grad()
def encode(model, x):
    """The frozen codes z, as a numpy array of shape (n, d)."""
    model.eval()
    ...          # <- replace this line


codes = {d: encode(models[d], x_val) for d in DIMS}
for d, z in codes.items():
    print(f"d = {d:<4d} -> z {z.shape}")

### 3.2 ✏️ · The two baselines

`evaluate_representation(z, y, name=..., verbose=False)` computes all four
numbers at once and returns one report. You will call it in every remaining session of this
course.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     one report for raw pixels, one PCA fitted to 32 components,
#     and one report for its transform of the validation set
# ──────────────────────────────────────────────────────────────────────
from sklearn.decomposition import PCA

# the raw-pixel baseline: evaluate x_val_np itself
raw = ...          # <- replace this line

# PCA with 32 components: fit on x_train_np, transform x_val_np, evaluate that
pca = ...          # <- replace this line
pca_rep = ...      # <- replace this line

print(raw.probe_accuracy, pca_rep.probe_accuracy)

### 3.3 ▶ · The table

In [ ]:
# Given - read it, run it, move on.
reports = [raw, pca_rep] + [
    evaluate_representation(codes[d], y_val, name=f"autoencoder d={d}", verbose=False)
    for d in DIMS
]

table = compare_reports(reports)
table

### 3.4 ✏️ · What mistakes does the probe make?

`probe_accuracy` is one number hiding ten classes. Fit the same probe yourself
and look at **which** classes it confuses - that tells you what the
representation failed to separate.

The fit is given; you produce the predictions and the matrix.

In [ ]:
# Given - read it, run it, move on.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler

z = codes[32]
cut = 4200                                   # same split the course helper uses
scaler = StandardScaler().fit(z[:cut])
clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(scaler.transform(z[:cut]), y_val[:cut])

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     predict on the held-back part, z[cut:] (remember the scaler),
#     then a confusion matrix against y_val[cut:], normalize="true"
# ──────────────────────────────────────────────────────────────────────
pred = ...          # <- replace this line
cm = ...            # <- replace this line

### 3.5 ▶ · Draw it

In [ ]:
# Given - read it, run it, move on.
fig, ax = plt.subplots(figsize=(6.4, 5.6))
im = ax.imshow(cm, cmap="Oranges", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.colorbar(im, fraction=0.046)
fig.tight_layout()
plt.show()

off = cm - np.diag(np.diag(cm))
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"worst confusion: {class_names[i]} predicted as {class_names[j]} "
      f"({off[i, j]:.0%} of the time)")

### ❓ Checkpoint 3

1. Which representations beat **raw pixels** on `probe_accuracy`? On
   `knn_accuracy`? **Are they the same ones?**
2. The autoencoder at $d = 8$ uses **98x fewer numbers** than raw pixels. How
   does it do on $k$-NN - and why is that the number a retrieval system cares
   about more than `probe_accuracy`?
3. For $d = 32$, compare `probe_accuracy` and `silhouette`. One is around 0.81,
   the other around 0.08. **Both are correct.** What is each telling you that
   the other is not? Name one product feature you could *not* build on this
   space despite the 0.81.
4. Look at the `ari` column. It comes from running $k$-means with the labels
   hidden and then checking how well the clusters line up with the truth.
   **Does any autoencoder beat raw pixels on it?** Reconcile that with $d = 32$
   clearly winning on $k$-NN - what are the two numbers measuring that makes
   them disagree?
5. What is the worst confusion in your matrix? Is it the same pair of classes
   that checkpoint 2 said was hardest to reconstruct? If not - what does that
   disagreement tell you about reconstruction error as a guide?
6. Going from $d = 32$ to $d = 128$ quadruples your storage bill. What did it
   buy? Which $d$ would you ship, and what would you need to know about the
   product to be sure?

*Your answers:*

1.
2.
3.
4.
5.
6.

---

# Checkpoint 4 - Wreck it with something irrelevant

**~20 minutes. The most important part of the lab.** One cell to write.

Real catalogue photos are not clean. Suppliers shoot under different lighting,
some images carry a watermark, some have a coloured corner. None of that says
anything about the product.

| | Nuisance | What it stands for |
|---|---|---|
| **A** | a 12x12 square of random brightness in the corner | a watermark, a badge, a supplier sticker |
| **B** | the whole image multiplied by a random factor in $[0.25, 1]$ | inconsistent studio lighting |

Neither one carries **any** information about the class. A perfect
representation would ignore them entirely.

### 4.1 ▶ · The two nuisances

In [ ]:
# Given - read it, run it, move on.
def add_patch(arr, rng, side=12):
    """Nuisance A: a constant-brightness square in the corner of each image."""
    out = arr.reshape(-1, 28, 28).copy()
    v = rng.uniform(0.0, 1.0, size=len(out)).astype(np.float32)
    out[:, 0:side, 0:side] = v[:, None, None]
    return out.reshape(len(out), -1)


def relight(arr, rng, lo=0.25, hi=1.0):
    """Nuisance B: every pixel of an image scaled by one random factor."""
    v = rng.uniform(lo, hi, size=len(arr)).astype(np.float32)
    return np.clip(arr * v[:, None], 0.0, 1.0)


NUISANCES = {"A - corner patch": add_patch, "B - lighting": relight}

### 4.2 ▶ · What they look like

In [ ]:
# Given - read it, run it, move on.
fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
for r, (name, fn) in enumerate(NUISANCES.items()):
    shown = fn(x_val_np[:8], np.random.default_rng(SEED))
    for c in range(8):
        axes[r][c].imshow(1 - shown[c].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
    axes[r][0].set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
fig.tight_layout()
plt.show()

### 4.3 ✏️ · Train on a corrupted catalogue

One function, five lines. **Nothing changes** except the data: same
architecture, same epochs, same seed.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 5 lines
#     train `m` for 8 epochs on train_np (torch.from_numpy(...copy()))
#     evaluate_representation on encode(m, <val as a tensor>) vs y_val
#     reconstruction MSE of m on val_np
#     return probe_accuracy, knn_accuracy, mse
# ──────────────────────────────────────────────────────────────────────
def run_on(train_np, val_np, d):
    """Train from scratch on `train_np`; measure on `val_np`. Returns 3 numbers."""
    set_seed(SEED)
    m = Autoencoder(d=d)
    ...          # <- replace this line
    return probe, knn, mse


print(run_on(x_train_np, x_val_np, 8))   # sanity check: close to the clean row

### 4.4 ▶ · Six models, about a minute

In [ ]:
# Given - read it, run it, move on.
import pandas as pd

rows = []
for d in (2, 8, 32):
    clean = evaluate_representation(codes[d], y_val, name="clean", verbose=False)
    rows.append({"d": d, "data": "clean", "probe": clean.probe_accuracy,
                 "knn": clean.knn_accuracy, "val mse": val_mse(models[d])})

    for name, fn in NUISANCES.items():
        tr = fn(x_train_np, np.random.default_rng(SEED))
        va = fn(x_val_np, np.random.default_rng(SEED + 1))
        probe, knn, mse = run_on(tr, va, d)
        rows.append({"d": d, "data": name, "probe": probe, "knn": knn,
                     "val mse": mse})

pd.DataFrame(rows).round(4)

### ❓ Checkpoint 4 - the questions that matter most today

1. Write out your table. At $d = 2$, how many points of `probe_accuracy` did
   the **corner patch** cost you? Express it as a share of the gap between the
   clean score and random guessing (0.10).
2. Now read the **`val mse` column** at $d = 8$ and $d = 32$. Compare the clean
   row to the two nuisance rows. **Which has the lower reconstruction error?**
3. That result should bother you. The models trained on corrupted data report a
   *better* loss and a *worse* representation. Give the explanation - and in
   doing so, answer this: **are those three MSE numbers even comparable?** What
   changed between them besides the model?
4. The nuisances carry no information about the product. Explain in two
   sentences why the encoder spends capacity on them anyway. *(What is the loss
   actually rewarded for? How many of the 784 pixels does a 12x12 square
   occupy?)*
5. Why is the damage much worse at $d = 2$ than at $d = 32$? What does that
   suggest about choosing $d$ when your real data is messy?
6. You must choose between two encoders for the assistant and all you are given
   is their reconstruction losses. **Write one sentence saying why that is not
   enough**, citing two numbers from your own table.
7. Name one thing you could change - in the data, the architecture or the loss
   - to make the encoder stop caring about the lighting. You do not have to
   implement it.

*Your answers:*

1.
2.
3.
4.
5.
6.
7.

---

# Checkpoint 5 - Find the hole

**~15 minutes.** Two cells to write, two lines each.

The latent space is good at *describing* products. Now try to *move around in
it*. Nothing in our loss ever said anything about the space **between** two
products.

### 5.1 ▶ · A decoder you can call directly

In [ ]:
# Given - read it, run it, move on.
@torch.no_grad()
def decode(model, z):
    model.eval()
    return model.decoder(torch.as_tensor(z, dtype=torch.float32)).numpy()


z32 = codes[32]
print(f"real codes: mean {z32.mean():+.2f}, std {z32.std():.2f}, "
      f"range [{z32.min():.1f}, {z32.max():.1f}]")

### 5.2 ✏️ · Decode eight points that are not products

The fairest possible test: draw them from a Gaussian with exactly the same
per-dimension mean and standard deviation as the real codes.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     z_random: rng.normal(z32.mean(0), z32.std(0), size=(8, 32)),
#                cast to float32
#     fakes:    decode them with models[32]
# ──────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
z_random = ...      # <- replace this line
fakes = ...         # <- replace this line

In [ ]:
# Given - read it, run it, move on.
fig, axes = plt.subplots(1, 8, figsize=(9, 1.4))
for c, ax in enumerate(axes):
    ax.imshow(1 - fakes[c].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Decoded from random points in the latent space", y=1.14)
plt.show()

### 5.3 ✏️ · Walk from one product to another

The point at fraction $t$ on the straight line between two codes is
$(1 - t)\,z_a + t\,z_b$.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line
#     the nine points on the line between z32[a] and z32[b],
#     stacked and decoded with models[32]
# ──────────────────────────────────────────────────────────────────────
a, b = 0, 3                                  # two validation products
alphas = np.linspace(0, 1, 9)
walk = ...          # <- replace this line

In [ ]:
# Given - read it, run it, move on.
fig, axes = plt.subplots(1, len(alphas), figsize=(len(alphas) * 1.1, 1.5))
for c, ax in enumerate(axes):
    ax.imshow(1 - walk[c].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{alphas[c]:.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"{class_names[y_val[a]]}  ->  {class_names[y_val[b]]}", y=1.2)
plt.show()

### ❓ Checkpoint 5

1. Are the random decodings **products**? Describe what you actually see.
2. In the walk, look at $t \approx 0.5$. Is it a plausible garment, or
   something else?
3. Our objective asked for exactly one thing. Write it in a sentence - then
   write what it **never asked for**, which is why this fails.
4. In one sentence: what extra instruction would you add to the loss so a
   random point decodes to something sensible?

> Question 4 is Friday's lecture. Write your answer before you hear mine.

*Your answers:*

1.
2.
3.
4.

---

# Checkpoint 6 - The challenge

**Open-ended. Most of you will not get here, and that is fine.**

> Build the best representation you can under a hard budget:
> **$d \leq 16$**, and **no more than 60 seconds of training** on your own
> machine. Maximise **`knn_accuracy`** on the clean validation set.

The reference to beat is a plain `Autoencoder(d=16)` trained for 8 epochs. It
scores about **0.82** in a few seconds. An hour of careful work gets you to
roughly **0.83** - the gains are small, which is exactly why *which lever moved
it* is the interesting answer, not the number.

| Lever | Try |
|---|---|
| epochs / learning rate | more epochs in the same 60 s; `lr` 3e-4 to 3e-3 |
| width and depth | wider hidden layer, or two hidden layers |
| activation | `ReLU`, `GELU`, `Tanh` |
| corruption | add noise to the input, keep the clean target |
| the loss | `L1Loss` instead of `MSELoss` |

**The rules.** Time only the training. Keep a row for **every** configuration
you try, including the ones that lose. One change at a time, or you will not
know which lever did it.

### 6.1 ▶ · The harness, given

In [ ]:
# Given - read it, run it, move on.
import time

results = []


def attempt(name, model, *, epochs, noise=0.0, lr=1e-3):
    """Train under the clock, evaluate, record the row."""
    set_seed(SEED)
    t0 = time.time()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    n = len(x_train)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, 256):
            clean = x_train[perm[i:i + 256]]
            inp = torch.clamp(clean + noise * torch.randn_like(clean), 0, 1) if noise else clean
            x_hat, _ = model(inp)
            loss = loss_fn(x_hat, clean)
            opt.zero_grad(); loss.backward(); opt.step()
    secs = time.time() - t0
    rep = evaluate_representation(encode(model, x_val), y_val, name=name, verbose=False)
    results.append({"config": name, "d": model.d, "seconds": round(secs, 1),
                    "knn": rep.knn_accuracy, "probe": rep.probe_accuracy})
    print(f"  {name:<30s} {secs:5.1f}s   knn={rep.knn_accuracy:.4f}  "
          f"probe={rep.probe_accuracy:.4f}")
    return rep


attempt("reference: d=16, 8 epochs", Autoencoder(d=16), epochs=8)

### 6.2 ✏️ · Your attempts

One `attempt(...)` call per configuration. Change **one thing** at a time.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about as many lines as you have ideas
#     one attempt(...) per configuration, one change at a time
# ──────────────────────────────────────────────────────────────────────
# attempt("d=16, 20 epochs", Autoencoder(d=16), epochs=20)
# attempt("d=16, 20 ep, hidden 512", Autoencoder(d=16, hidden=512), epochs=20)
# ... keep going, and keep the losers

In [ ]:
# Given - read it, run it, move on.
import pandas as pd

pd.DataFrame(results).sort_values("knn", ascending=False)

### ❓ Checkpoint 6

1. Your best `knn_accuracy` under the budget, and the configuration that got
   it.
2. **Which single lever mattered most?** How do you know - which two rows of
   your table prove it?
3. Did anything you expected to help actually hurt? Why do you think that is?
4. Your best model has $d \leq 16$. How does it compare to the $d = 32$ and
   $d = 128$ models from checkpoint 3? What does that say about how much the
   *width* of the bottleneck really mattered?

*Your answers:*

1.
2.
3.
4.

---

## What you built today

- An encoder that turns a product into a handful of numbers, trained **without
  a single label**.
- The measurement loop - **train, freeze, probe, compare** - that every
  remaining session of this course runs on.
- Direct evidence that **reconstruction loss is not a proxy for usefulness**:
  two nuisances that carry no information made the loss look *better* and the
  representation measurably *worse*.
- A latent space that reads well and travels badly.

**Friday:** the variational autoencoder, which closes the holes you found in
checkpoint 5.